In [3]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import os  # NEW

# ---------- symbolic definitions ----------
x_sym = sp.symbols('x')
f_sym = x_sym**3 - 7*x_sym**2 + 14*x_sym - 5
fprime_sym = sp.diff(f_sym, x_sym)

# numeric functions
f = sp.lambdify(x_sym, f_sym, 'numpy')
fprime = sp.lambdify(x_sym, fprime_sym, 'numpy')
# ----- compute real root(s) and critical points -----
roots_sym = sp.nroots(f_sym)
real_roots = [complex(r) for r in roots_sym if abs(sp.im(r)) < 1e-10]
real_roots = [r.real for r in real_roots]

crit_points_sym = sp.nroots(sp.Eq(fprime_sym, 0))
real_crit_points = [complex(c) for c in crit_points_sym if abs(sp.im(c)) < 1e-10]
real_crit_points = [c.real for c in real_crit_points]

def newton_raphson_single(x0, max_iter=50, precision=1e-8):
    x = float(x0)

    xs = [x]
    fs_vals = [f(x)]
    fps_vals = [fprime(x)]
    iters_used = 0
    converged = False

    # (iteration index, x_k, f(x_k), f'(x_k), relative error)
    details = [(0, x, f(x), fprime(x), None)]

    for k in range(max_iter):
        fx = f(x)
        fpx = fprime(x)
        if fpx == 0:
            iters_used = k + 1
            break

        x_new = x - fx / fpx
        fx_new = f(x_new)
        fpx_new = fprime(x_new)

        rel_err = abs(x_new - x) / max(abs(x_new), 1e-16)

        xs.append(x_new)
        fs_vals.append(fx_new)
        fps_vals.append(fpx_new)
        iters_used = k + 1
        details.append((k + 1, x_new, fx_new, fpx_new, rel_err))

        if rel_err < precision:
            converged = True
            x = x_new
            break

        x = x_new

    return {
        "x0": x0,
        "xs": xs,
        "fs": fs_vals,
        "fps": fps_vals,
        "iters": iters_used,
        "converged": converged,
        "details": details
    }
def main():
    np.random.seed(0)

    # directory to save all figures
    save_dir = os.getcwd()
    print(f"Plots will be saved to: {save_dir}")

    # initial guesses: 0 to 10 with step 0.01
    x0_values = np.arange(0, 10 + 0.01, 0.01)
    max_iter = int(input("Enter max number of iterations: "))

    results = []
    for x0 in x0_values:
        res = newton_raphson_single(x0, max_iter=max_iter, precision=1e-8)
        results.append(res)

    # -------- up to 3 non-convergent examples --------
    nonconv_indices = [i for i, r in enumerate(results) if not r["converged"]]
    special_nonconv_indices = nonconv_indices[:3]

    print("\n=== Up to three examples that do NOT converge (if available) ===")
    for idx in special_nonconv_indices:
        r = results[idx]
        print(f"x0 = {r['x0']:.2f}, iterations = {r['iters']}, converged = {r['converged']}")

    # -------- choose 10 cases: 3 lowest iters, 3 highest, 4 random from rest --------
    total_points = len(results)
    all_iters = np.array([r["iters"] for r in results])
    all_indices = np.arange(total_points)

    sorted_by_iters = np.argsort(all_iters)
    lowest3 = sorted_by_iters[:3]
    highest3 = sorted_by_iters[-3:]
    remaining = np.setdiff1d(all_indices, np.concatenate((lowest3, highest3)))
    num_random = min(4, len(remaining))
    random4 = np.random.choice(remaining, size=num_random, replace=False) if num_random > 0 else np.array([], dtype=int)

    chosen_indices = np.concatenate((lowest3, highest3, random4))

    # root label for legend
    root_label = None
    if real_roots:
        root_label = f"Root ≈ {real_roots[0]:.6f}"

    # -------- detailed plots (function + derivative per case) --------
    for case_num, idx in enumerate(chosen_indices, start=1):
        res = results[idx]
        xs = res["xs"]
        fs_vals = res["fs"]
        x0 = res["x0"]
        details = res["details"]

        print(f"\n=== Newton iteration details for x0 = {x0:.2f} ===")
        for (k, xk, fxk, fpxk, errk) in details:
            if errk is None:
                print(f"Iter {k}: x = {xk}, f(x) = {fxk}, f'(x) = {fpxk}, error = None (initial guess)")
            else:
                print(f"Iter {k}: x = {xk}, f(x) = {fxk}, f'(x) = {fpxk}, error = {errk}")
        print(f"Total iterations for x0 = {x0:.2f}: {res['iters']}")
        print(f"Converged: {res['converged']}")

        # --- plot 1: function with iterates, root, critical points ---
        x_min = min(xs + real_roots + real_crit_points) - 1
        x_max = max(xs + real_roots + real_crit_points) + 1
        x_plot = np.linspace(x_min, x_max, 400)
        y_plot = f(x_plot)

        plt.figure(figsize=(6, 4))
        plt.axhline(0, color='black', linewidth=0.5, zorder=0)
        plt.plot(x_plot, y_plot, label='f(x)', zorder=1)

        plt.scatter(xs, fs_vals, color='red', s=40, zorder=3, label='Newton iterates')

        x_init = xs[0]
        f_init = fs_vals[0]
        plt.scatter(x_init, f_init, color='blue', s=80, marker='s',
                    zorder=4, label='Initial point')

        first_root = True
        for r in real_roots:
            lbl = root_label if first_root and root_label is not None else ""
            plt.scatter(r, f(r), color='green', s=70, marker='x', zorder=5, label=lbl)
            first_root = False

        first_cp = True
        for c in real_crit_points:
            plt.scatter(c, f(c), color='purple', s=70, marker='^',
                        zorder=5, label='Critical point' if first_cp else "")
            first_cp = False

        plt.xlabel('$x_k$')
        plt.ylabel('f(x)')
        plt.legend()
        plt.grid(True, zorder=0)
        plt.title(f"Newton iteration starting at $x_0$ = {x0:.2f}")

        # save + show
        filename = f"newton_case_{case_num}_fx.png"
        save_path = os.path.join(save_dir, filename)
        plt.savefig(save_path, dpi=300)
        print(f"Saved plot: {save_path}")
        plt.show()

        # --- plot 2: derivative as a function of x_k (scatter) ---
        x_values = [d[1] for d in details]   # x_k
        fp_values = [d[3] for d in details]  # f'(x_k)

        plt.figure(figsize=(6, 4))
        plt.scatter(x_values, fp_values, marker='o', color='black')
        plt.xlabel(r'$x_{k}$ (iterate)')
        plt.ylabel(r"$f'(x_{k})$")
        plt.title(rf"Derivative vs $x_{{k}}$ ($x_0$ = {x0:.2f})")
        plt.grid(True)

        filename = f"newton_case_{case_num}_derivative.png"
        save_path = os.path.join(save_dir, filename)
        plt.savefig(save_path, dpi=300)
        print(f"Saved plot: {save_path}")
        plt.show()
